# 3.2 Arquitectura del Data Warehouse: El Esquema de Estrella (Banca)
Vamos a simular un Data Warehouse en memoria. Tenemos una **Tabla de Hechos** (las transacciones crudas) rodeada de **Tablas de Dimensión** (el contexto de la transacción). Nuestro objetivo de KDD es crear una matriz plana para alimentar a un modelo de riesgo.

In [9]:
import pandas as pd

print("--- 1. TABLAS DE DIMENSIÓN (El Contexto) ---")
# Dimensión Cliente (Desnormalizada: el segmento ya viene escrito, no es otro ID)
dim_clientes = pd.DataFrame({
    'id_cliente': [101, 102, 103, 104],
    'estado': ['BC', 'CDMX', 'BC', 'Jalisco'],
    'segmento': ['Universitario', 'Premium', 'Universitario', 'Standard']
})

display(dim_clientes)

# Dimensión Tiempo
dim_tiempo = pd.DataFrame({
    'id_tiempo': [20260914, 20260915],
    'dia_semana': ['Lunes', 'Martes'],
    'es_quincena': [False, True]
})
display(dim_tiempo)

print("\n--- 2. TABLA DE HECHOS (La Métrica) ---")
# La tabla de hechos solo une IDs numéricos y la métrica (Monto)
fact_transacciones = pd.DataFrame({
    'id_transaccion': [1, 2, 3, 4, 5],
    'id_cliente': [101, 102, 101, 104, 103],
    'id_tiempo': [20260914, 20260914, 20260915, 20260915, 20260915],
    'monto_mxn': [150.50, 4500.00, 85.00, 1200.00, 300.00]
})


display(fact_transacciones)

--- 1. TABLAS DE DIMENSIÓN (El Contexto) ---


,id_cliente,estado,segmento
0,101,BC,Universitario
1,102,CDMX,Premium
2,103,BC,Universitario
3,104,Jalisco,Standard


,id_tiempo,dia_semana,es_quincena
0,20260914,Lunes,False
1,20260915,Martes,True



--- 2. TABLA DE HECHOS (La Métrica) ---


,id_transaccion,id_cliente,id_tiempo,monto_mxn
0,1,101,20260914,150.5
1,2,102,20260914,4500.0
2,3,101,20260915,85.0
3,4,104,20260915,1200.0
4,5,103,20260915,300.0


### Ejecutando la Tubería ETL en Memoria
Para que el modelo de Machine Learning entienda esto, debemos pasar por las tres fases de la arquitectura de datos.

In [10]:
print("--- INICIANDO PIPELINE ETL ---")

# EXTRACT & TRANSFORM: Cruzamos los hechos con las dimensiones (Desnormalización masiva)
matriz_kdd = pd.merge(fact_transacciones, dim_clientes, on='id_cliente', how='inner')
matriz_kdd = pd.merge(matriz_kdd, dim_tiempo, on='id_tiempo', how='inner')

# TRANSFORM: Purgamos llaves primarias operativas que causan ruido matemático
matriz_kdd = matriz_kdd.drop(columns=['id_transaccion', 'id_cliente', 'id_tiempo'])

# LOAD: La matriz final se "carga" en la variable de memoria lista para la IA
print("--- MATRIZ MULTIDIMENSIONAL (LOAD COMPLETO) ---")
display(matriz_kdd)

--- INICIANDO PIPELINE ETL ---
--- MATRIZ MULTIDIMENSIONAL (LOAD COMPLETO) ---


,monto_mxn,estado,segmento,dia_semana,es_quincena
0,150.5,BC,Universitario,Lunes,False
1,4500.0,CDMX,Premium,Lunes,False
2,85.0,BC,Universitario,Martes,True
3,1200.0,Jalisco,Standard,Martes,True
4,300.0,BC,Universitario,Martes,True


---
### 🛠️ Zona de Experimentación y Reto de Código

En los Almacenes de Datos, la operación analítica clásica se llama **Roll-up** (Acumular o Resumir datos subiendo de granularidad). 

**Reto 1: Código Analítico**
Usa Pandas para responder a la pregunta que tiró el servidor de producción: *¿Cuál es el monto total de dinero transaccionado por cada Segmento de Cliente?*
Escribe el código usando la función `.groupby()` sobre tu `matriz_kdd`.

In [11]:
# ESCRIBE TU CÓDIGO AQUÍ:
# Pista: Agrupa por 'segmento', selecciona 'monto_mxn' y aplica .sum()

resultado_rollup = matriz_kdd.groupby('segmento')['monto_mxn'].sum()
print(resultado_rollup)

segmento
Premium          4500.0
Standard         1200.0
Universitario     535.5
Name: monto_mxn, dtype: float64


**Reto 2: Arquitectura en Papel (Responde aquí mismo)**
Observa nuestra tabla `dim_clientes`. En una base de datos relacional estricta (OLTP), el segmento "Universitario" sería un número (un ID_Segmento) que apuntaría a otra tabla de catálogo. 
En el Almacén de Datos (OLAP) decidimos escribir "Universitario" directamente en texto, repitiéndolo por cada cliente. 
Como ingeniero de software: ¿Por qué estamos violando las reglas de normalización a propósito en esta arquitectura? ¿Qué ganamos al aceptar esa redundancia?

> **Tu respuesta:** Estamos repitiendo algunos datos para hacer las consultas más rápidas y sencillas. En un Data Warehouse buscamos analizar grandes cantidades de información, no tanto evitar la redundancia como en una base de datos transaccional.

> Al guardar directamente el segmento en la tabla de clientes evitamos hacer más uniones con otras tablas, lo que facilita los análisis y mejora el rendimiento.


## Actividad de reforzamiento: Clínicas dentales

### **1. Tabla de Hechos: Fact_Segmentaciones_Dentales**

Cada fila representa una visita en la que se analizó una radiografía con IA.

**Métricas numéricas:**
- Porcentaje_Bacterias: porcentaje detectado por la IA.
- Tiempo_Inferencia_IA_Seg: tiempo que tardó el algoritmo.
- conteo_visitas: valor 1 por visita para poder contarlas.

**Llaves foráneas:**
- id_paciente: conecta con Dim_Pacientes.
- id_sucursal: conecta con Dim_Sucursales.
- id_tiempo: conecta con Dim_Tiempo.

También guardaremos ID_Visita como identificador de cada visita, pero no se utilizará como una métrica.



### **2. Tablas de dimensión**

**Dim_Pacientes**
- id_paciente (PK)
- Edad_Paciente
- rango_edad

Nos permite clasificar las visitas según la edad del paciente.

**Dim_Sucursales**
- id_sucursal (PK)
- Sucursal_Atencion
- Ciudad

Sirve para identificar en qué clínica se realizó cada visita.

**Dim_Tiempo**
- id_tiempo (PK)
- Fecha_Consulta
- Dia_Semana
- mes
- anio
- es_fin_semana

Permite filtrar los registros por mes y saber cuáles fueron durante el fin de semana.


In [18]:

# Datos originales de las clínicas dentales
registros = [
    [8001, 'Raúl Soto', 45, 'Clínica Centro', 'Ensenada', '2026-10-03', 'Sábado', 68.5, 1.2, 'iVBORw0KGgoAAAANSU...'],
    [8002, 'Sofía Lira', 12, 'Clínica Valle', 'Tijuana', '2026-10-03', 'Sábado', 15.0, 0.8, 'iVBORw0KGgoAAAANSU...'],
    [8003, 'Raúl Soto', 45, 'Clínica Centro', 'Ensenada', '2026-10-04', 'Domingo', 65.2, 1.1, 'iVBORw0KGgoAAAANSU...'],
    [8004, 'Ana Cruz', 68, 'Clínica Sur', 'Ensenada', '2026-10-06', 'Martes', 82.1, 2.5, 'iVBORw0KGgoAAAANSU...'],
    [8005, 'Luis Rey', 25, 'Clínica Valle', 'Tijuana', '2026-10-06', 'Martes', 5.4, 0.9, 'iVBORw0KGgoAAAANSU...']
]

columnas = [
    'ID_Visita', 'Paciente', 'Edad_Paciente',
    'Sucursal_Atencion', 'Ciudad', 'Fecha_Consulta',
    'Dia_Semana', 'Porcentaje_Bacterias',
    'Tiempo_Inferencia_IA_Seg', 'Mascara_IA_Base64'
]

datos_crudos = pd.DataFrame(registros, columns=columnas)

display(datos_crudos)


,ID_Visita,Paciente,Edad_Paciente,Sucursal_Atencion,Ciudad,Fecha_Consulta,Dia_Semana,Porcentaje_Bacterias,Tiempo_Inferencia_IA_Seg,Mascara_IA_Base64
0,8001,Raúl Soto,45,Clínica Centro,Ensenada,2026-10-03,Sábado,68.5,1.2,iVBORw0KGgoAAAANSU...
1,8002,Sofía Lira,12,Clínica Valle,Tijuana,2026-10-03,Sábado,15.0,0.8,iVBORw0KGgoAAAANSU...
2,8003,Raúl Soto,45,Clínica Centro,Ensenada,2026-10-04,Domingo,65.2,1.1,iVBORw0KGgoAAAANSU...
3,8004,Ana Cruz,68,Clínica Sur,Ensenada,2026-10-06,Martes,82.1,2.5,iVBORw0KGgoAAAANSU...
4,8005,Luis Rey,25,Clínica Valle,Tijuana,2026-10-06,Martes,5.4,0.9,iVBORw0KGgoAAAANSU...


In [19]:

# Copiamos los datos para transformarlos
df_dental = datos_crudos.copy()

# Convertimos la fecha a formato de fecha
df_dental['Fecha_Consulta'] = pd.to_datetime(df_dental['Fecha_Consulta'])

# Dividimos las edades en rangos
df_dental['rango_edad'] = pd.cut(
    df_dental['Edad_Paciente'],
    bins=[-1, 17, 39, 59, 120],
    labels=['0-17', '18-39', '40-59', '60+']
)

# Creamos el identificador de fecha y detectamos fines de semana
df_dental['id_tiempo'] = df_dental['Fecha_Consulta'].dt.strftime('%Y%m%d').astype(int)
df_dental['es_fin_semana'] = df_dental['Fecha_Consulta'].dt.dayofweek >= 5

display(df_dental[['Paciente', 'rango_edad', 'es_fin_semana']])


,Paciente,rango_edad,es_fin_semana
0,Raúl Soto,40-59,True
1,Sofía Lira,0-17,True
2,Raúl Soto,40-59,True
3,Ana Cruz,60+,False
4,Luis Rey,18-39,False


In [20]:

# Dimensión Pacientes
dim_pacientes = df_dental[
    ['Paciente', 'Edad_Paciente', 'rango_edad']
].drop_duplicates().reset_index(drop=True)

dim_pacientes.insert(0, 'id_paciente', range(1, len(dim_pacientes) + 1))

# Dimensión Sucursales
dim_sucursales = df_dental[
    ['Sucursal_Atencion', 'Ciudad']
].drop_duplicates().reset_index(drop=True)

dim_sucursales.insert(0, 'id_sucursal', range(1, len(dim_sucursales) + 1))

# Dimensión Tiempo
dim_tiempo_dental = df_dental[
    ['id_tiempo', 'Fecha_Consulta', 'Dia_Semana', 'es_fin_semana']
].drop_duplicates().reset_index(drop=True)

dim_tiempo_dental['mes'] = dim_tiempo_dental['Fecha_Consulta'].dt.month
dim_tiempo_dental['anio'] = dim_tiempo_dental['Fecha_Consulta'].dt.year

display(dim_pacientes)
display(dim_sucursales)
display(dim_tiempo_dental)


,id_paciente,Paciente,Edad_Paciente,rango_edad
0,1,Raúl Soto,45,40-59
1,2,Sofía Lira,12,0-17
2,3,Ana Cruz,68,60+
3,4,Luis Rey,25,18-39


,id_sucursal,Sucursal_Atencion,Ciudad
0,1,Clínica Centro,Ensenada
1,2,Clínica Valle,Tijuana
2,3,Clínica Sur,Ensenada


,id_tiempo,Fecha_Consulta,Dia_Semana,es_fin_semana,mes,anio
0,20261003,2026-10-03,Sábado,True,10,2026
1,20261004,2026-10-04,Domingo,True,10,2026
2,20261006,2026-10-06,Martes,False,10,2026


In [21]:

# Relacionamos las visitas con las dimensiones
fact_segmentaciones_dentales = (
    df_dental
    .merge(dim_pacientes[['Paciente', 'id_paciente']], on='Paciente')
    .merge(dim_sucursales[['Sucursal_Atencion', 'id_sucursal']], on='Sucursal_Atencion')
)

# Seleccionamos solo IDs y métricas
fact_segmentaciones_dentales = fact_segmentaciones_dentales[
    [
        'ID_Visita', 'id_paciente', 'id_sucursal',
        'id_tiempo', 'Porcentaje_Bacterias',
        'Tiempo_Inferencia_IA_Seg'
    ]
].copy()

fact_segmentaciones_dentales['conteo_visitas'] = 1

display(fact_segmentaciones_dentales)


,ID_Visita,id_paciente,id_sucursal,id_tiempo,Porcentaje_Bacterias,Tiempo_Inferencia_IA_Seg,conteo_visitas
0,8001,1,1,20261003,68.5,1.2,1
1,8002,2,2,20261003,15.0,0.8,1
2,8003,1,1,20261004,65.2,1.1,1
3,8004,3,3,20261006,82.1,2.5,1
4,8005,4,2,20261006,5.4,0.9,1


In [22]:

# Unimos la tabla de hechos con las dimensiones
reporte = (
    fact_segmentaciones_dentales
    .merge(dim_pacientes[['id_paciente', 'rango_edad']], on='id_paciente')
    .merge(dim_sucursales[['id_sucursal', 'Sucursal_Atencion']], on='id_sucursal')
    .merge(dim_tiempo_dental[['id_tiempo', 'es_fin_semana', 'mes', 'anio']], on='id_tiempo')
)

# Solo fines de semana de octubre 2026
reporte = reporte[
    (reporte['es_fin_semana']) &
    (reporte['mes'] == 10) &
    (reporte['anio'] == 2026)
]

# Promedio de bacterias por rango de edad y clínica
resultado = (
    reporte.groupby(
        ['rango_edad', 'Sucursal_Atencion'],
        observed=True
    )
    .agg(
        promedio_bacterias=('Porcentaje_Bacterias', 'mean'),
        visitas=('conteo_visitas', 'sum')
    )
    .reset_index()
    .sort_values('promedio_bacterias', ascending=False)
)

display(resultado)


,rango_edad,Sucursal_Atencion,promedio_bacterias,visitas
1,40-59,Clínica Centro,66.85,2
0,0-17,Clínica Valle,15.00,1



**Resultado del análisis**

Con los registros disponibles, la Clínica Centro en Ensenada presenta el mayor porcentaje promedio de bacterias durante el fin de semana, con 66.85% en pacientes de 40 a 59 años.

Esto se obtuvo de 2 visitas del mismo paciente. Por lo tanto, todavía necesitaríamos más registros para identificar una tendencia general.



### 3. Pregunta sobre ETL

Sería un error guardar las máscaras Base64 dentro de la Tabla de Hechos porque cada registro ocuparía demasiado espacio. Si tenemos millones de visitas, la base de datos crecería muchísimo y las consultas serían más lentas y costosas.

**Mi solución usando ETL:**

- **Extract:** obtener los datos de la aplicación web.
- **Transform:** separar la máscara Base64 de los datos numéricos, convertirla en un archivo de imagen y guardar solamente su referencia.
- **Load:** cargar al Data Warehouse las métricas y dimensiones necesarias para los análisis.

Las imágenes se guardarían en un almacenamiento externo seguro, como Azure Blob Storage, y se podrían recuperar usando su identificador.

Así evitamos llenar el Data Warehouse con datos pesados que no necesitamos para calcular los porcentajes de bacterias.
